### 🛠️ Configure Infrastructure Parameters & Create the Infrastructure

Set your desired parameters for the AG-APIM-PE infrastructure deployment.

❗️ **Modify entries under _User-defined parameters_**.

🔐 **Important**: After deployment, you'll need to install a self-signed certificate to avoid SSL warnings when testing HTTPS endpoints. This can be done using the Python cell below.

**Note:** This infrastructure includes Application Gateway with API Management using private endpoints. The creation process includes two phases: initial deployment with public access, private link approval, and then disabling public access.

In [1]:
import utils
from apimtypes import *

# ------------------------------
#    USER CONFIGURATION
# ------------------------------

rg_location = 'eastus2'             # Azure region for deployment
index       = 30                     # Infrastructure index (use different numbers for multiple environments)
apim_sku    = APIM_SKU.STANDARDV2   # Options: 'STANDARDV2', 'PREMIUMV2' (Basic is not supported for private endpoints)



# ------------------------------
#    SYSTEM CONFIGURATION
# ------------------------------

inb_helper = utils.InfrastructureNotebookHelper(rg_location, INFRASTRUCTURE.AG_APIM_PE, index, apim_sku) 
inb_helper.create_infrastructure()

utils.print_ok('All done!')


🚀 Creating infrastructure...

   Infrastructure : ag-apim-pe
   Index          : 30
   Resource group : apim-infra-ag-apim-pe-30
   Location       : eastus2
   APIM SKU       : Standardv2

📁 Changed working directory to: C:\Dev\Azure-Samples\Apim-Samples\infrastructure\ag-apim-pe
📝 Updated the policy XML in the bicep parameters file 'params.json'
👉🏽 Creating the resource group now...

✅ Infrastructure creation completed successfully!
📁 Restored working directory to: c:\Dev\Azure-Samples\Apim-Samples

✅ All done! ⌚ 18:02:15.440041 


### 🔐 Install Self-Signed Certificate (Required for HTTPS Testing)

⚠️ **This step is required to avoid SSL certificate warnings when testing Application Gateway HTTPS endpoints.**

The deployment creates an HTTPS-only Application Gateway with a self-signed certificate. To browse to the endpoints without SSL warnings, install this certificate in your local trust store using the Python cell below.

**Certificate Details:**
- Certificate name: `apim-samples-{resourceGroupName}`
- Stored securely in Azure Key Vault
- Cross-platform installation support

In [ ]:
# Install the self-signed certificate to your local trust store
utils.print_info("🔐 Installing self-signed certificate...")

success = utils.install_certificate_for_infrastructure(
    INFRASTRUCTURE.AG_APIM_PE, 
    index,
    inb_helper.rg_name
)

if success:
    utils.print_ok("✅ Certificate installation completed!")
    utils.print_info("You can now browse to HTTPS endpoints without SSL warnings.")
else:
    utils.print_error("❌ Certificate installation failed. Manual installation instructions were provided above.")

👉🏽 🔐 Installing self-signed certificate... 
👉🏽 Installing certificate for ag-apim-pe infrastructure (index 30) 
👉🏽 Resource Group: apim-infra-ag-apim-pe-30 
ℹ️ Looking for Key Vault in resource group... ⌚ 18:03:30.433226 
⚙️ az keyvault list -g apim-infra-ag-apim-pe-30 --query "[0].name" -o tsv 

✅ Found Key Vault: kv-pzb6inardekvi ⌚ 18:03:33.396749 
ℹ️ Downloading certificate 'ag-cert' from Key Vault... ⌚ 18:03:33.396749 
⚙️ az keyvault secret download --vault-name kv-pzb6inardekvi --name ag-cert --file C:\Users\SIMONK~1\AppData\Local\Temp\tmpf8ohz4zv.pfx --overwrite 

✅ Found Key Vault: kv-pzb6inardekvi ⌚ 18:03:33.396749 
ℹ️ Downloading certificate 'ag-cert' from Key Vault... ⌚ 18:03:33.396749 
⚙️ az keyvault secret download --vault-name kv-pzb6inardekvi --name ag-cert --file C:\Users\SIMONK~1\AppData\Local\Temp\tmpf8ohz4zv.pfx --overwrite 
⛔ Command failed with error: ERROR: (Forbidden) Caller is not authorized to perform action on resource.

If role assignments, deny assignments or

Traceback (most recent call last):
  File "C:\Dev\Azure-Samples\Apim-Samples\shared\python\utils.py", line 1847, in run
    output_text = subprocess.check_output(command, shell = True, stderr = subprocess.STDOUT).decode('utf-8')
                  ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\simonkurtz\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 466, in check_output
    return run(*popenargs, stdout=PIPE, timeout=timeout, check=True,
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\simonkurtz\AppData\Local\Programs\Python\Python312\Lib\subprocess.py", line 571, in run
    raise CalledProcessError(retcode, process.args,
subprocess.CalledProcessError: Command 'az keyvault secret download --vault-name kv-pzb6inardekvi --name ag-cert --file C:\Users\SIMONK~1\AppData\Local\Temp\tmpf8ohz4zv.pfx --overwrite' returned non-zero exit status 1.


⚙️ az account show --query id -o tsv 
⚙️ az role assignment create --assignee "744cffd5-e99d-4cc0-9fe3-2d284e07a1c4" --role "Key Vault Secrets User" --scope "/subscriptions/5fb73327-9152-4f64-bf8a-90dc0cc4ad8f/resourcegroups/apim-infra-ag-apim-pe-30/providers/Microsoft.KeyVault/vaults/kv-pzb6inardekvi" 

✅ Successfully granted Key Vault Secrets User access to current user ⌚ 18:03:47.537714 
👉🏽 Retrying certificate download... 
⚙️ az keyvault secret download --vault-name kv-pzb6inardekvi --name ag-cert --file C:\Users\SIMONK~1\AppData\Local\Temp\tmpf8ohz4zv.pfx --overwrite 

✅ Certificate downloaded (3448 bytes) ⌚ 18:03:55.243027 
ℹ️ Installing certificate to windows trust store... ⌚ 18:03:55.246028 
ℹ️ Certificate name: apim-samples-apim-infra-ag-apim-pe-30 ⌚ 18:03:55.246028 


### 🗑️ Clean up resources

When you're finished experimenting, it's advisable to remove all associated resources from Azure to avoid unnecessary cost.
Use the [clean-up notebook](clean-up.ipynb) for that.